# `reader` (ReaderModule)
- **Category**: Output (Agentic LLM)
- **Role**: 확장된 원본 셀 다큐먼트 리스트를 프롬프트에 주입하고, LangChain BaseTool 기반 수학 계산 및 메타데이터 조회 도구를 결합하여 환각 없는 최종 답변을 합성합니다.


In [ ]:
import sys
from pathlib import Path
import json

# 프로젝트 루트 경로 등록
PROJECT_ROOT = Path(".").resolve().parent.parent if Path(".").resolve().name == "modules" else Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

def print_io(title: str, input_data: dict, output_data: dict):
    print("=" * 70)
    print(f"📌 [Module Execution] {title}")
    print("=" * 70)
    print("\n📥 [Input DTO]")
    print(json.dumps(input_data, indent=2, ensure_ascii=False))
    print("\n📤 [Output Result]")
    print(json.dumps(output_data, indent=2, ensure_ascii=False))
    print("\n")


In [ ]:
from unittest.mock import MagicMock
from modules.reader.reader import ReaderModule, ReaderInputDTO, ReaderConfigDTO
from backend.providers.llm.chat_completion import ChatCompletionResult

mock_reader_llm = MagicMock()
mock_reader_llm.complete_with_metadata.return_value = ChatCompletionResult(
    content="2023년 삼성전자의 영업이익은 65,670억원이며, 2022년(433,766억원) 대비 약 84.86% 감소하였습니다. [Sheet: 손익계산서 | Cell: C5, D5]",
    usage={"prompt_tokens": 320, "completion_tokens": 75, "total_tokens": 395},
    latency_seconds=1.12,
)

module = ReaderModule(completion_client=mock_reader_llm)

sample_input = {
    "context_json": {
        "query_context": {
            "question_id": "QUERY-001",
            "question_text": "2023년 삼성전자 영업이익과 2022년 대비 증감율은 얼마인가요?"
        },
        "document_context": {
            "file_name": "samsung_2023.xlsx",
            "workbook_hash": "hash_samsung_2023"
        },
        "items": [
            "Company: 삼성전자 | Sheet: 손익계산서 | Row Header: 영업이익 | Column Header: 2021 | Cell Value: 516339",
            "Company: 삼성전자 | Sheet: 손익계산서 | Row Header: 영업이익 | Column Header: 2022 | Cell Value: 433766",
            "Company: 삼성전자 | Sheet: 손익계산서 | Row Header: 영업이익 | Column Header: 2023 | Cell Value: 65670"
        ]
    }
}
input_dto = ReaderInputDTO(**sample_input)
output = module.run(input_dto, config=ReaderConfigDTO(model="gpt-5.6-luna", enable_tools=True))
print_io("reader (ReaderModule)", sample_input, output)
